# 01 — Coleta e integração dos dados da PRF

Baixa os arquivos anuais de acidentes agrupados por ocorrência (2020–2025), consolida os registros e salva a camada inicial em Parquet.

In [ ]:
import os
import zipfile
import pandas as pd
import gdown

PASTA_PROJETO = "TCC_PRF"
PASTA_RAW = os.path.join(PASTA_PROJETO, "data", "raw")
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
os.makedirs(PASTA_RAW, exist_ok=True)
os.makedirs(PASTA_PROCESSED, exist_ok=True)

arquivos_prf = {
    2020: "1esu6IiH5TVTxFoedv6DBGDd01Gvi8785",
    2021: "12xH8LX9aN2gObR766YN3cMcuycwyCJDz",
    2022: "1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-",
    2023: "1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq",
    2024: "14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO",
    2025: "1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR"
}

In [ ]:
for ano, file_id in arquivos_prf.items():
    pasta_ano = os.path.join(PASTA_RAW, str(ano))
    os.makedirs(pasta_ano, exist_ok=True)
    arquivo_zip = os.path.join(pasta_ano, f"datatran{ano}.zip")
    if not os.path.exists(arquivo_zip):
        print(f"Baixando base PRF {ano}...")
        gdown.download(f"https://drive.google.com/uc?id={file_id}", arquivo_zip, quiet=False)

for ano in arquivos_prf:
    pasta_ano = os.path.join(PASTA_RAW, str(ano))
    arquivo_zip = os.path.join(pasta_ano, f"datatran{ano}.zip")
    if zipfile.is_zipfile(arquivo_zip):
        with zipfile.ZipFile(arquivo_zip, "r") as z:
            z.extractall(pasta_ano)
        print(f"{ano} descompactado.")

In [ ]:
def ler_csv_prf(caminho):
    for encoding in ["utf-8", "latin1", "ISO-8859-1"]:
        try:
            return pd.read_csv(caminho, sep=";", encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            pass
    raise UnicodeDecodeError("encoding", b"", 0, 1, f"Falha ao ler {caminho}")

bases = []
for ano in arquivos_prf:
    pasta_ano = os.path.join(PASTA_RAW, str(ano))
    csvs = [f for f in os.listdir(pasta_ano) if f.lower().endswith(".csv")]
    if not csvs:
        continue
    df_ano = ler_csv_prf(os.path.join(pasta_ano, csvs[0]))
    df_ano.columns = df_ano.columns.str.strip().str.lower()
    df_ano["ano_fonte"] = ano
    print(f"{ano}: {df_ano.shape[0]:,} registros | {df_ano.shape[1]} colunas")
    bases.append(df_ano)

df_prf = pd.concat(bases, ignore_index=True, sort=False)
print(f"Base consolidada: {df_prf.shape[0]:,} linhas e {df_prf.shape[1]} colunas")
print(df_prf["ano_fonte"].value_counts().sort_index())

In [ ]:
def converter_decimal_br(serie):
    return pd.to_numeric(
        serie.astype("string").str.strip().str.replace(",", ".", regex=False),
        errors="coerce"
    )

for coluna in ["latitude", "longitude", "km"]:
    if coluna in df_prf.columns:
        df_prf[coluna] = converter_decimal_br(df_prf[coluna])

print("IDs duplicados:", df_prf["id"].duplicated().sum())

arquivo_saida = os.path.join(PASTA_PROCESSED, "prf_acidentes_2020_2025.parquet")
df_prf.to_parquet(arquivo_saida, index=False, engine="pyarrow")
print("Salvo em:", arquivo_saida)